# 🤟 Wasel v4 Pro — Live Sign Language Translator
### OpenPose Keypoints + PSL Kaggle Dataset + Live Webcam

**Pipeline:**
1. Cell 1: Install dependencies
2. Cell 2: Download Kaggle PSL dataset & train classifier
3. Cell 3: Setup live engine (MediaPipe for feature extraction)
4. Cell 4: Launch live translator with public URL

> ⚡ **Trained on real PSL data — translates in real-time!**

In [ ]:
#@title 📦 Cell 1: Install Dependencies

print("⏳ Installing dependencies...")

!pip install -q \
    "gradio>=5.0.0" \
    "mediapipe>=0.10.0" \
    "scikit-learn>=1.3.0" \
    "opencv-python-headless" \
    "opendatasets"

print("\n🔍 Verifying...")
import importlib
for name, mod in {"gradio": "gradio", "mediapipe": "mediapipe", "sklearn": "sklearn", "cv2": "cv2"}.items():
    try:
        m = importlib.import_module(mod)
        print(f"   ✅ {name}: {getattr(m, '__version__', '?')}")
    except Exception as e:
        print(f"   ❌ {name}: {e}")

print("\n🎉 Ready! Proceed to Cell 2.")

In [ ]:
#@title 🗄️ Cell 2: Download PSL Dataset & Train Classifier
#@markdown Downloads the Pakistan Sign Language OpenPose dataset from Kaggle.
#@markdown You need a Kaggle account. Get your credentials from:
#@markdown https://www.kaggle.com/settings → API → Create New Token

KAGGLE_USERNAME = "" #@param {type:"string"}
KAGGLE_KEY = "" #@param {type:"string"}

import os, json, sqlite3, pickle
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# ─── Setup Kaggle credentials ───
if KAGGLE_USERNAME and KAGGLE_KEY:
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    kaggle_json = {'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}
    with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
        json.dump(kaggle_json, f)
    os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
    print("   ✅ Kaggle credentials saved.")
else:
    print("   ⚠️ No Kaggle credentials. Will try alternative download...")

# ─── Download dataset ───
DATASET_DIR = "/content/psl_dataset"
DB_PATH = None

if not os.path.exists(DATASET_DIR):
    print("⏳ Downloading PSL dataset from Kaggle...")
    try:
        !kaggle datasets download -d saadbutt32/pakistan-sign-language-dataset -p {DATASET_DIR} --unzip -q
        print("   ✅ Dataset downloaded.")
    except Exception as e:
        print(f"   ❌ Kaggle download failed: {e}")
        print("   Trying alternative...")
        import opendatasets as od
        od.download('https://www.kaggle.com/datasets/saadbutt32/pakistan-sign-language-dataset', DATASET_DIR)
else:
    print("   ✅ Dataset already downloaded.")

# ─── Find the database file ───
for root, dirs, files in os.walk(DATASET_DIR):
    for f in files:
        if f.endswith('.db'):
            DB_PATH = os.path.join(root, f)
            break

if DB_PATH:
    print(f"   ✅ Found database: {DB_PATH}")
else:
    # Try to find JSON files instead
    json_count = 0
    for root, dirs, files in os.walk(DATASET_DIR):
        json_count += len([f for f in files if f.endswith('.json')])
    print(f"   📁 Found {json_count} JSON files (no .db file)")

# ─── Load data from SQLite database ───
X_data = []
y_data = []

if DB_PATH:
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    # List tables
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
    tables = [t[0] for t in cursor.fetchall()]
    print(f"   📊 Tables in database: {tables}")

    # Use poseDataset (has body + both hands)
    target_table = 'poseDataset' if 'poseDataset' in tables else tables[0]
    print(f"   📊 Using table: {target_table}")

    # Get schema
    cursor.execute(f"PRAGMA table_info({target_table})")
    columns = cursor.fetchall()
    col_names = [c[1] for c in columns]
    print(f"   📊 Columns ({len(col_names)}): {col_names[:5]}...{col_names[-3:]}")

    # Load all data
    cursor.execute(f"SELECT * FROM {target_table}")
    rows = cursor.fetchall()
    print(f"   📊 Total samples: {len(rows)}")

    # Find label column (usually last)
    label_col_idx = col_names.index('label') if 'label' in col_names else len(col_names) - 1
    feature_indices = [i for i in range(len(col_names)) if i != label_col_idx and col_names[i] != 'id']

    for row in rows:
        features = [float(row[i]) if row[i] is not None else 0.0 for i in feature_indices]
        label = str(row[label_col_idx])
        X_data.append(features)
        y_data.append(label)

    conn.close()
else:
    # Fallback: Load from JSON files
    print("   ⏳ Loading from JSON files...")
    for root, dirs, files in os.walk(DATASET_DIR):
        label = os.path.basename(root)
        for f in files:
            if f.endswith('.json'):
                try:
                    with open(os.path.join(root, f), 'r') as jf:
                        data = json.load(jf)
                    if 'people' in data and len(data['people']) > 0:
                        person = data['people'][0]
                        features = []
                        for key in ['pose_keypoints_2d', 'hand_left_keypoints_2d', 'hand_right_keypoints_2d']:
                            if key in person:
                                features.extend(person[key])
                        if features:
                            X_data.append(features)
                            y_data.append(label)
                except:
                    pass

X = np.array(X_data)
y = np.array(y_data)
print(f"\n📊 Dataset loaded: {X.shape[0]} samples, {X.shape[1]} features per sample")
print(f"📊 Unique labels ({len(np.unique(y))}): {list(np.unique(y)[:10])}...")

# ─── Train classifier ───
print("\n⏳ Training classifier...")
le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"   ✅ Accuracy: {acc*100:.1f}%")
print(f"   ✅ Features expected: {clf.n_features_in_}")
print(f"   ✅ Classes: {list(le.classes_)}")

# Save classifier
MODEL_PATH = "/content/psl_classifier_openpose.pkl"
with open(MODEL_PATH, 'wb') as f:
    pickle.dump((clf, le), f)
print(f"\n💾 Classifier saved to {MODEL_PATH}")
print(f"\n🎉 Training complete! Proceed to Cell 3.")

In [ ]:
#@title 🧠 Cell 3: Setup Live Engine
#@markdown Uses MediaPipe Holistic for live feature extraction.
#@markdown Features are mapped to match the trained classifier.

import cv2, pickle, logging
import numpy as np
import mediapipe as mp
from collections import deque

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("wasel")

# ─── Load trained classifier ───
MODEL_PATH = "/content/psl_classifier_openpose.pkl"
with open(MODEL_PATH, 'rb') as f:
    classifier, label_encoder = pickle.load(f)
N_FEATURES = classifier.n_features_in_
print(f"   ✅ Classifier loaded: {N_FEATURES} features, {len(label_encoder.classes_)} classes")
print(f"   🏷️ Classes: {list(label_encoder.classes_)}")

# ─── MediaPipe Holistic Setup ───
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
holistic = mp_holistic.Holistic(
    static_image_mode=False,
    model_complexity=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)
print("   ✅ MediaPipe Holistic loaded.")

# ─── Feature Extraction ───
def extract_features(results):
    """Extract features from MediaPipe and format to match the classifier."""
    features = []

    # Body pose (use first 13 landmarks to match OpenPose body)
    if results.pose_landmarks:
        # Map MediaPipe pose to OpenPose-compatible format
        for lm in results.pose_landmarks.landmark[:13]:
            features.extend([lm.x, lm.y])
    else:
        features.extend([0.0] * 26)

    # Left hand (21 landmarks)
    if results.left_hand_landmarks:
        for lm in results.left_hand_landmarks.landmark:
            features.extend([lm.x, lm.y])
    else:
        features.extend([0.0] * 42)

    # Right hand (21 landmarks)
    if results.right_hand_landmarks:
        for lm in results.right_hand_landmarks.landmark:
            features.extend([lm.x, lm.y])
    else:
        features.extend([0.0] * 42)

    # Pad or trim to match classifier's expected features
    feat = np.array(features)
    if len(feat) > N_FEATURES:
        feat = feat[:N_FEATURES]
    elif len(feat) < N_FEATURES:
        feat = np.pad(feat, (0, N_FEATURES - len(feat)))

    return feat

# ─── Frame Processing ───
sequence_buffer = deque(maxlen=15)

def process_frame(image):
    """Process a frame: detect pose + hands, predict sign."""
    if image is None:
        return image
    try:
        rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = holistic.process(rgb)
        annotated = image.copy()

        # Draw pose skeleton
        if results.pose_landmarks:
            mp_drawing.draw_landmarks(
                annotated, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS,
                mp_drawing_styles.get_default_pose_landmarks_style())

        # Draw hand skeletons (the finger bones!)
        if results.left_hand_landmarks:
            mp_drawing.draw_landmarks(
                annotated, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
                mp_drawing_styles.get_default_hand_landmarks_style())
        if results.right_hand_landmarks:
            mp_drawing.draw_landmarks(
                annotated, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
                mp_drawing_styles.get_default_hand_landmarks_style())

        # Extract features and predict
        label_text = "Waiting for signs..."
        feat = extract_features(results)
        sequence_buffer.append(feat)

        if len(sequence_buffer) >= 3:
            avg_feat = np.mean(np.array(list(sequence_buffer)), axis=0).reshape(1, -1)
            probs = classifier.predict_proba(avg_feat)[0]
            idx = np.argmax(probs)
            conf = float(probs[idx]) * 100
            if conf > 25.0:
                label = label_encoder.inverse_transform([idx])[0]
                label_text = f"{label} ({conf:.1f}%)"

        # Draw big green text
        cv2.putText(annotated, label_text, (20, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3)

        return annotated
    except Exception as e:
        logger.error(f"Frame error: {e}")
        return image

print("\n🚀 Live engine ready! Proceed to Cell 4.")

In [ ]:
#@title 🎥 Cell 4: Launch Live Translator
#@markdown Opens webcam with real-time PSL translation.
#@markdown A public `*.gradio.live` URL will appear below!

import gradio as gr

demo = gr.Interface(
    fn=process_frame,
    inputs=gr.Image(sources=["webcam"], streaming=True, label="📸 Your Camera"),
    outputs=gr.Image(label="🤟 Wasel v4 Live Translation"),
    live=True,
    title="🤟 Wasel v4 Pro — Pakistan Sign Language Translator",
    description="Show PSL signs to the camera. The AI detects your body, hands, and fingers, then translates the sign in real-time!",
)

print("🎉 Launching Wasel v4 Live Demo...")
print("📋 A public URL will appear below — share it!")
print("⚠️  Keep this cell running.\n")

demo.launch(share=True, quiet=False)